In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [2]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhuwa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\bhuwa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhuwa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
rmv_verb=WordNetLemmatizer()
def clean(text):
    text=text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text=re.sub(r'[^\w\s]',"",text)
    words=text.split()
    words=[rmv_verb.lemmatize(word,pos='v')for word in words]
    return " ".join(words)

In [ ]:
ds = load_dataset("milistu/AMAZON-Products-2023")
df=pd.DataFrame(ds['train'].select_columns(['title', 'description', 'main_category']))
df['text']=(df['title']+' '+df['description'])

In [ ]:
df['text']=df['text'].apply(clean)

In [6]:
df.head()

,title,description,main_category
0,anomie bonhomie,amazoncom fan of scritti polittis synthpopfunk...,digital music
1,sunshine on my shoulder the best of john denve...,sunshine on my shoulder be a 2cd 36track colle...,digital music
2,18 greatest hit of 38 special,track listings 1 rockin into the night 2 hold ...,digital music
3,the gift cd,second studio album by the multimillionselling...,digital music
4,τηε βοοτlεg sεrιεs vοι ᛐ7 ᛐ996ᛐ997 frαԍμεντտ τ...,eu edition 2cd disc one time out of mind 2022 ...,digital music


In [ ]:
cmp_df=df[df['main_category'].notna()]
msng_df=df[df['main_category'].isna()]
cmp_df['main_category']=cmp_df['main_category'].str.lower()

In [9]:
vc = cmp_df['main_category'].value_counts()
cmp_df['main_category'] = cmp_df['main_category'].apply(
    lambda x: x if vc[x] >= 50 else "other")
x_train,x_test,y_train,y_test=train_test_split(cmp_df['text'],cmp_df['main_category'],test_size=0.2,random_state=42,stratify=cmp_df['main_category'])

C:\Users\bhuwa\AppData\Local\Temp\ipykernel_1548\1221402977.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cmp_df['main_category'] = cmp_df['main_category'].apply(


In [10]:
lr_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('lr',LogisticRegression(max_iter=2000,class_weight="balanced"))])
nb_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('nb',MultinomialNB())])
svm_model=Pipeline([("cng_vctr",TfidfVectorizer(max_features=50000,ngram_range=(1,2),min_df=2,max_df=0.9,stop_words="english")),
                ('svm',LinearSVC(class_weight="balanced"))])

In [11]:
lr_model.fit(x_train,y_train)
nb_model.fit(x_train,y_train)
svm_model.fit(x_train,y_train)

Pipeline(steps=[('cng_vctr',
                 TfidfVectorizer(max_df=0.9, max_features=50000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english')),
                ('svm', LinearSVC(class_weight='balanced'))])

In [12]:
lr_pred=lr_model.predict(x_test)
nb_pred=nb_model.predict(x_test)
svm_pred=svm_model.predict(x_test)

In [13]:
print("...........lr_report............\n")
print(classification_report(y_test,lr_pred))


...........lr_report............

                           precision    recall  f1-score   support

               all beauty       0.10      0.38      0.16        13
          all electronics       0.76      0.76      0.76       627
           amazon fashion       0.97      0.89      0.93      7292
              amazon home       0.91      0.76      0.83      2775
               appliances       0.38      0.87      0.53        62
     appstore for android       0.74      0.82      0.78        28
    arts, crafts & sewing       0.45      0.81      0.58       175
               automotive       0.83      0.87      0.85      1196
           camera & photo       0.73      0.88      0.80       168
          car electronics       0.44      0.50      0.47        14
cell phones & accessories       0.92      0.87      0.89       694
  collectibles & fine art       1.00      0.91      0.95        23
                computers       0.85      0.88      0.87       345
            digital music  

In [14]:
print("\n\n..........nb_report...........\n")
print(classification_report(y_test,nb_pred))



..........nb_report...........

                           precision    recall  f1-score   support

               all beauty       0.00      0.00      0.00        13
          all electronics       0.63      0.71      0.67       627
           amazon fashion       0.88      0.98      0.93      7292
              amazon home       0.61      0.92      0.73      2775
               appliances       0.00      0.00      0.00        62
     appstore for android       0.00      0.00      0.00        28
    arts, crafts & sewing       0.94      0.18      0.30       175
               automotive       0.79      0.85      0.82      1196
           camera & photo       0.86      0.18      0.30       168
          car electronics       0.00      0.00      0.00        14
cell phones & accessories       0.89      0.87      0.88       694
  collectibles & fine art       0.00      0.00      0.00        23
                computers       0.90      0.74      0.82       345
            digital music  

C:\Users\bhuwa\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bhuwa\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bhuwa\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [15]:
print("\n\n.........svm_report........\n")
print(classification_report(y_test,svm_pred))



.........svm_report........

                           precision    recall  f1-score   support

               all beauty       0.38      0.38      0.38        13
          all electronics       0.78      0.80      0.79       627
           amazon fashion       0.97      0.96      0.97      7292
              amazon home       0.91      0.85      0.88      2775
               appliances       0.55      0.77      0.64        62
     appstore for android       0.83      0.71      0.77        28
    arts, crafts & sewing       0.65      0.73      0.69       175
               automotive       0.84      0.89      0.87      1196
           camera & photo       0.79      0.86      0.82       168
          car electronics       0.67      0.57      0.62        14
cell phones & accessories       0.93      0.91      0.92       694
  collectibles & fine art       0.95      0.87      0.91        23
                computers       0.84      0.89      0.87       345
            digital music     

In [16]:
msng_df['text']=(msng_df['title']+' '+msng_df['description'])

C:\Users\bhuwa\AppData\Local\Temp\ipykernel_1548\3071722513.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msng_df['text']=(msng_df['title']+' '+msng_df['description'])


In [17]:
msng_df['main_category']=svm_model.predict(msng_df['text'])

C:\Users\bhuwa\AppData\Local\Temp\ipykernel_1548\2437530035.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msng_df['main_category']=svm_model.predict(msng_df['text'])


In [18]:
msng_df['main_category'].value_counts()

main_category
amazon fashion               8812
amazon home                  5904
cell phones & accessories    1532
sports & outdoors            1434
health & personal care       1232
all electronics              1142
handmade                      866
pet supplies                  816
computers                     579
tools & home improvement      571
industrial & scientific       507
arts, crafts & sewing         436
office products               294
automotive                    246
appliances                    129
video games                    75
toys & games                   49
camera & photo                 48
digital music                  31
other                          29
musical instruments            22
all beauty                     15
home audio & theater           13
appstore for android            8
gps & navigation                7
car electronics                 7
collectibles & fine art         1
Name: count, dtype: int64

In [19]:
cmp_df=cmp_df.drop(['text'],axis=1)
msng_df=msng_df.drop(['text'],axis=1)

In [20]:
cmp_df.head()

,title,description,main_category
0,anomie bonhomie,amazoncom fan of scritti polittis synthpopfunk...,digital music
1,sunshine on my shoulder the best of john denve...,sunshine on my shoulder be a 2cd 36track colle...,digital music
2,18 greatest hit of 38 special,track listings 1 rockin into the night 2 hold ...,digital music
3,the gift cd,second studio album by the multimillionselling...,digital music
4,τηε βοοτlεg sεrιεs vοι ᛐ7 ᛐ996ᛐ997 frαԍμεντտ τ...,eu edition 2cd disc one time out of mind 2022 ...,digital music


In [21]:
msng_df.head()

,title,description,main_category
178,hyuoep guitar chord chart poster reference cir...,learn guitar scale the easy way with this hand...,musical instruments
226,geesatis guitar pick box case organizer 1 pcs ...,convenient and high quality geesatis guitar pi...,musical instruments
231,puloa bass fretboard chart metal sign guitar p...,our store provide many retro style design with...,office products
411,universal bass control knob bass volume knob w...,this amplifier bass volume control knob be spe...,musical instruments
433,ikkegol upgrade 2023 usb foot pedal switch opt...,plug and play and program by our software it c...,video games


In [22]:
final_df=pd.concat([cmp_df,msng_df],axis=0)

In [23]:
final_df.head()

,title,description,main_category
0,anomie bonhomie,amazoncom fan of scritti polittis synthpopfunk...,digital music
1,sunshine on my shoulder the best of john denve...,sunshine on my shoulder be a 2cd 36track colle...,digital music
2,18 greatest hit of 38 special,track listings 1 rockin into the night 2 hold ...,digital music
3,the gift cd,second studio album by the multimillionselling...,digital music
4,τηε βοοτlεg sεrιεs vοι ᛐ7 ᛐ996ᛐ997 frαԍμεντտ τ...,eu edition 2cd disc one time out of mind 2022 ...,digital music


In [ ]:
final_df.to_csv("clean_data.csv",index=False)